# Mini-RQ 2: Architektur oder Embedding?

**Deep Learning · Master Wirtschaftsinformatik · HHN · WS 2026/27**

| | |
|---|---|
| **Team** | *Namen eintragen* |
| **Abgabe** | Woche 7, So 15.11.2026, 23:59 |
| **Aufgabenstellung** | `RQ2_Architektur_Embedding` (ILIAS) |

---

## Wozu dieses Notebook da ist

Dieses Notebook ist ein **Gerüst, keine Vorlage zum Ausfüllen**. Es enthält nur das, was für alle Teams gleich sein muss, damit die Ergebnisse vergleichbar sind:

1. **Reproduzierbarkeit:** Seeds setzen, Versionen und Gerät ausgeben
2. **Daten:** AG News (Thema) und Rotten Tomatoes (Sentiment) laden und fest aufteilen
3. **Tokenisierung und Vokabular:** damit alle Modelle dieselben Eingaben sehen
4. **Embeddings:** GloVe laden und die drei Varianten (zufällig, GloVe fest, GloVe mittrainiert) bauen
5. **Ergebnisschema:** die Liste, in die jeder Messwert kommt
6. **Teil B:** eine Hilfsfunktion für die Auswahlen von k Seeds
7. **Export** und **Abgabecheckliste**

Alles andere schreibt ihr selbst: die drei Modelle (Baseline, TextCNN, LSTM), die Trainingsschleife, die Metriken, die Heatmaps, die Auswertung von Teil B und die Diskussion.

## 1. Reproduzierbarkeit

Neuronale Netze starten mit zufälligen Gewichten, und auch das Mischen der Batches ist zufällig. Ein **Seed** legt den Zufallsgenerator fest: gleicher Seed, gleiches Ergebnis.

**Ruft `set_seed` am Anfang jedes einzelnen Laufs** mit dem Seed dieses Laufs auf, nicht nur einmal oben im Notebook. Laut Aufgabenstellung braucht Teil A mindestens 3 Seeds je Konfiguration, Teil B mindestens 10 Seeds je Architektur. Die Seeds 1 bis 3 aus Teil A zählen in Teil B mit, benutzt also **dieselben Seed-Nummern 1 bis 10**.

Die Zelle enthält außerdem `count_params`, mit dem ihr die Parameterzahl eurer Modelle für den Bericht bestimmt.

In [ ]:
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn


def set_seed(seed: int) -> None:
    """Fix all random number generators used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Number of parameters of a model (for the report)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


# CPU is sufficient for this assignment. A GPU is used automatically if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python : {sys.version.split()[0]}")
print(f"NumPy  : {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {device}")

## 2. Daten

| Aufgabe | Datensatz | Klassen | Zufallsniveau |
|---|---|---|---|
| `thema` | **AG News**: Nachrichtenmeldungen (Titel + Beschreibung), 120 000 Trainings- und 7 600 Testbeispiele | World, Sports, Business, Sci/Tech | 25 % |
| `sentiment` | **Rotten Tomatoes**: Filmkritik-Sätze, mitgelieferte Aufteilung in Train, Validation und Test | negativ, positiv | 50 % |

Die Zelle lädt beide Datensätze über die Bibliothek `datasets` (beim ersten Mal Download in den Hugging-Face-Cache) und erzeugt für jede Aufgabe ein Dictionary mit `train`, `val` und `test`. Jeder Teil ist ein Paar `(Texte, Labels)`.

**Was ihr festlegt:** `N_TRAIN_THEMA` und `N_VAL_THEMA`. AG News ist groß, für das Rechenbudget (unter einer Stunde, ohne GPU) nehmt ihr nur einen Teil. Wählt die Größen einmal, **begründet sie im Bericht und ändert sie danach nicht mehr**. Sie gelten für alle Modelle, Embeddings und Seeds. Der Teilsatz wird mit dem festen `SPLIT_SEED` gezogen und ist damit für alle Teams mit denselben Größen identisch.

In [ ]:
from datasets import load_dataset

SPLIT_SEED = 2026         # fixed for all teams - do not change
N_TRAIN_THEMA = 20_000    # your choice, justify in the report, then keep it fixed
N_VAL_THEMA = 2_000       # your choice, justify in the report, then keep it fixed


def to_pair(split):
    """Hugging Face split -> (list of texts, list of int labels)."""
    return list(split["text"]), list(split["label"])


def subset(pair, n, rng):
    """Random subset of size n of a (texts, labels) pair."""
    idx = rng.permutation(len(pair[0]))[:n]
    return [pair[0][i] for i in idx], [pair[1][i] for i in idx]


# --- Task "thema": AG News (4 classes) ---
ag = load_dataset("ag_news")
ag_train_full = to_pair(ag["train"])
ag_test = to_pair(ag["test"])
rng = np.random.RandomState(SPLIT_SEED)
perm = rng.permutation(len(ag_train_full[0]))
val_idx, train_idx = perm[:N_VAL_THEMA], perm[N_VAL_THEMA:N_VAL_THEMA + N_TRAIN_THEMA]
ag_val = ([ag_train_full[0][i] for i in val_idx], [ag_train_full[1][i] for i in val_idx])
ag_train = ([ag_train_full[0][i] for i in train_idx], [ag_train_full[1][i] for i in train_idx])

# --- Task "sentiment": Rotten Tomatoes (2 classes), split as provided ---
rt = load_dataset("rotten_tomatoes")

TASKS = {
    "thema": {"train": ag_train, "val": ag_val, "test": ag_test, "n_classes": 4},
    "sentiment": {
        "train": to_pair(rt["train"]),
        "val": to_pair(rt["validation"]),
        "test": to_pair(rt["test"]),
        "n_classes": 2,
    },
}

for name, task in TASKS.items():
    sizes = {part: len(task[part][0]) for part in ("train", "val", "test")}
    print(f"{name:10s} classes={task['n_classes']}  {sizes}")
    print("   example:", task["train"][0][0][:80].replace("\n", " "), "->", task["train"][1][0])

## 3. Tokenisierung und Vokabular

Damit alle Modelle dieselben Eingaben sehen, ist die Vorverarbeitung fest: Kleinschreibung, Zerlegung in Wörter und Zahlen, ein Vokabular **nur aus dem Trainingsteil** (Wörter, die mindestens `MIN_FREQ`-mal vorkommen) und Padding bzw. Abschneiden auf `MAX_LEN` Token. Index 0 ist `<PAD>`, Index 1 ist `<UNK>` (Wörter, die nicht im Vokabular stehen).

`MAX_LEN` ist je Aufgabe vorgegeben: AG News hat längere Texte als die Sentiment-Sätze. Die Zelle erzeugt für jede Aufgabe die Tensoren `X_train`, `X_val`, `X_test` (Form `[N, MAX_LEN]`) und `y_train`, `y_val`, `y_test`.

In [ ]:
import re
from collections import Counter

PAD, UNK = 0, 1
MIN_FREQ = 2
MAX_LEN = {"thema": 60, "sentiment": 40}   # fixed for all teams - do not change


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9']+", text.lower())


def build_vocab(texts, min_freq=MIN_FREQ) -> dict:
    counts = Counter(tok for t in texts for tok in tokenize(t))
    vocab = {"<PAD>": PAD, "<UNK>": UNK}
    for tok, c in counts.most_common():
        if c >= min_freq:
            vocab[tok] = len(vocab)
    return vocab


def encode(texts, vocab, max_len) -> torch.Tensor:
    """List of texts -> LongTensor [N, max_len], padded with 0 and truncated."""
    out = torch.zeros(len(texts), max_len, dtype=torch.long)
    for i, t in enumerate(texts):
        ids = [vocab.get(tok, UNK) for tok in tokenize(t)][:max_len]
        out[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    return out


DATA = {}   # task -> dict with vocab, X_*, y_*, n_classes
for name, task in TASKS.items():
    vocab = build_vocab(task["train"][0])
    d = {"vocab": vocab, "n_classes": task["n_classes"]}
    for part in ("train", "val", "test"):
        d[f"X_{part}"] = encode(task[part][0], vocab, MAX_LEN[name])
        d[f"y_{part}"] = torch.tensor(task[part][1], dtype=torch.long)
    DATA[name] = d
    print(f"{name:10s} vocab={len(vocab):,}  X_train={tuple(d['X_train'].shape)}  X_test={tuple(d['X_test'].shape)}")

## 4. Embeddings

Alle drei Varianten haben **50 Dimensionen**, damit sich nur die Herkunft der Vektoren unterscheidet:

| Variante | `variante` | Startwerte | wird mit dem Modell trainiert? |
|---|---|---|---|
| zufällig | `"zufaellig"` | zufällig | ja |
| GloVe fest | `"glove_fest"` | GloVe | nein (eingefroren) |
| GloVe mittrainiert | `"glove_mittrainiert"` | GloVe | ja |

Die Funktion `make_embedding(variante, vocab, glove_matrix)` gibt eine fertige `nn.Embedding`-Schicht zurück. Ihr baut sie **in jedem Modell an der Stelle ein, an der sonst `nn.Embedding` stünde**. `padding_idx=0` sorgt dafür, dass der Vektor für `<PAD>` null bleibt und keinen Gradienten bekommt.

Wörter, die im Vokabular, aber nicht in GloVe stehen, bekommen einen kleinen zufälligen Vektor (gleiche Streuung wie GloVe). Die Zelle gibt aus, wie viel Prozent des Vokabulars GloVe abdeckt. Diese Zahl gehört in den Bericht, weil sie zwischen den beiden Aufgaben stark abweichen kann.

In [ ]:
import gensim.downloader as api

EMB_DIM = 50
glove = api.load("glove-wiki-gigaword-50")   # first call downloads ca. 70 MB
assert glove.vector_size == EMB_DIM
GLOVE_STD = float(np.std(glove.vectors))


def build_glove_matrix(vocab: dict, seed: int = SPLIT_SEED) -> torch.Tensor:
    """Matrix [len(vocab), 50]: GloVe vector if known, small random vector otherwise, zeros for <PAD>."""
    g = torch.Generator().manual_seed(seed)
    matrix = torch.randn(len(vocab), EMB_DIM, generator=g) * GLOVE_STD
    for tok, idx in vocab.items():
        if tok in glove.key_to_index:
            matrix[idx] = torch.tensor(glove[tok])
    matrix[PAD] = 0.0
    return matrix


def make_embedding(variante: str, vocab: dict, glove_matrix: torch.Tensor) -> nn.Embedding:
    """Embedding layer for one of the three variants. Call set_seed(seed) BEFORE this for "zufaellig"."""
    if variante == "zufaellig":
        emb = nn.Embedding(len(vocab), EMB_DIM, padding_idx=PAD)
    elif variante == "glove_fest":
        emb = nn.Embedding.from_pretrained(glove_matrix.clone(), freeze=True, padding_idx=PAD)
    elif variante == "glove_mittrainiert":
        emb = nn.Embedding.from_pretrained(glove_matrix.clone(), freeze=False, padding_idx=PAD)
    else:
        raise ValueError(f"unknown variant: {variante}")
    return emb


GLOVE_MATRIX = {}
for name, d in DATA.items():
    GLOVE_MATRIX[name] = build_glove_matrix(d["vocab"])
    covered = sum(tok in glove.key_to_index for tok in d["vocab"]) / len(d["vocab"])
    print(f"{name:10s} GloVe covers {covered:.1%} of the vocabulary")

## 5. Ergebnisschema

Jeder einzelne Messwert wird als **eine Zeile** in der Liste `results` abgelegt, als Dictionary mit genau diesen sieben Spalten:

| Spalte | Bedeutung | Beispiel |
|---|---|---|
| `run_id` | eindeutige Kennung des Laufs | `"thema-lstm-glovefest_s1"` |
| `konfiguration` | Aufgabe, Modell und Embedding | `"aufgabe=thema \| modell=lstm \| embedding=glove_fest"` |
| `seed` | verwendeter Seed (1 bis 10 in Teil B) | `1` |
| `split` | auf welchem Teil gemessen | `"test"` oder `"val"` |
| `metrik` | Name der Metrik | `"accuracy"`, `"macro_f1"` |
| `wert` | Messwert | `0.874` |
| `laufzeit_s` | Trainingszeit des Laufs in Sekunden | `23.4` |

Ein Lauf mit zwei Metriken auf dem Testset ergibt also zwei Zeilen mit derselben `run_id`. Zusätzliche Spalten bitte nicht anlegen. Wenn ihr mehr festhalten wollt, gehört es in die `konfiguration`.

`add_result` hängt eine Zeile an und prüft dabei das Schema. Die Zeilen aus **Teil B** (Auswahlen von k Seeds) gehören **nicht** in `results`, sie sind eine Auswertung der Läufe und kein eigener Lauf.

In [ ]:
RESULT_COLUMNS = ["run_id", "konfiguration", "seed", "split", "metrik", "wert", "laufzeit_s"]

results = []   # one dict per measured value, keys = RESULT_COLUMNS


def add_result(run_id, konfiguration, seed, split, metrik, wert, laufzeit_s):
    """Append one measured value in the required schema."""
    if split not in ("test", "val"):
        raise ValueError("split must be 'test' or 'val'")
    results.append({
        "run_id": run_id, "konfiguration": konfiguration, "seed": int(seed),
        "split": split, "metrik": metrik, "wert": float(wert), "laufzeit_s": float(laufzeit_s),
    })


# Example of one call (delete or ignore): two rows for one run
# add_result("thema-lstm-glovefest_s1", "aufgabe=thema | modell=lstm | embedding=glove_fest", 1, "test", "accuracy", 0.874, 23.4)

## 6. Teil B: Auswahlen von k Seeds

Für Teil B habt ihr pro Architektur **10 Test-Accuracies** (Seeds 1 bis 10). Die Funktion `subset_means(values, k)` gibt die Mittelwerte **aller möglichen Auswahlen** von k dieser Werte zurück: bei 10 Werten sind das 10, 45, 120, 210, 252, 210, 120, 45 und 10 Mittelwerte für k = 1 bis 9. Ein Wert zählt in einer Auswahl höchstens einmal, und es braucht keinen Zufall.

Daraus bildet ihr selbst, je k und Modell, den Durchschnitt und das Band (5. bis 95. Perzentil), und zeichnet beides gegen k. Das Beispiel unten benutzt **erfundene** Zahlen und dient nur dazu, die Funktion auszuprobieren.

In [ ]:
from itertools import combinations


def subset_means(values, k: int) -> np.ndarray:
    """Means of all possible selections of k values out of `values` (without replacement)."""
    return np.array([np.mean(c) for c in combinations(values, k)])


# Demo with INVENTED numbers - replace by your own 10 test accuracies per architecture
demo = [80, 82, 79, 83, 81, 80, 82, 78, 81, 80]
for k in (1, 3, 5):
    m = subset_means(demo, k)
    print(f"k={k}: {len(m):3d} selections, mean={m.mean():.1f}, 5-95% band=({np.percentile(m, 5):.1f}, {np.percentile(m, 95):.1f})")

---

## Ab hier: eure Experimente

Hier beginnt eure Arbeit. Gliedert sie so, dass jemand ohne Vorwissen folgen kann. Kurze Markdown-Zellen, die sagen, was als Nächstes passiert und warum, sind Teil der Bewertung („Dokumentation und Lesbarkeit“).

Ein sinnvoller Ablauf:

1. **Erwartung** in einer Markdown-Zelle festhalten, bevor ihr den ersten Lauf startet
2. die drei **Modelle** definieren (Baseline, TextCNN, LSTM), jeweils mit `make_embedding` als Embedding-Schicht
3. eine **Funktion für einen Lauf**: `set_seed`, Modell bauen, trainieren, auf Validation und Test messen, Zeit stoppen, `add_result` aufrufen
4. erst **ein** Lauf zum Testen, dann die Schleife über Aufgaben, Architekturen, Embeddings und Seeds (Teil A)
5. Teil B: die Heatmap-Spalte (Aufgabe und Embedding) wählen, in der die beiden besten Architekturen am dichtesten beieinander liegen, und dort die Seeds 4 bis 10 nachlegen
6. Heatmaps, Teil-B-Abbildung und Diskussion

Rechnet die Laufzeit vorab hoch: Ein Lauf auf einer Stichprobe zeigt, wie viele von den mindestens 75 Läufen in die Stunde passen. Passt es nicht, verkleinert ihr `N_TRAIN_THEMA`, die Epochenzahl oder die Modellgröße, **nie** die Zahl der Seeds.

---

## 7. Export

Schreibt alle gesammelten Ergebnisse als CSV neben das Notebook. Die Zelle prüft vorher, ob jede Zeile genau die sieben Spalten hat. Die CSV wird mit abgegeben.

In [ ]:
import csv

RESULTS_FILE = "rq2_results.csv"

bad_rows = [i for i, row in enumerate(results) if set(row) != set(RESULT_COLUMNS)]
if bad_rows:
    raise ValueError(f"Rows {bad_rows[:5]} do not match the result schema {RESULT_COLUMNS}")

with open(RESULTS_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
    writer.writeheader()
    writer.writerows(results)

print(f"{len(results)} rows written to {RESULTS_FILE}")

## 8. Abgabecheckliste

- [ ] Teamnamen oben eingetragen
- [ ] **Restart & Run All** läuft ohne Fehler durch
- [ ] `set_seed` wird zu Beginn jedes Laufs aufgerufen, Teil A hat mindestens 3 Seeds je Konfiguration, Teil B 10 Seeds je Architektur
- [ ] Alle 18 Konfigurationen (3 Architekturen × 3 Embedding-Varianten × 2 Aufgaben) sind gemessen
- [ ] Pro Architektur dieselben Hyperparameter über alle Embeddings und Aufgaben, Auswahl nur auf Validation, nie auf dem Testset
- [ ] `N_TRAIN_THEMA` und `N_VAL_THEMA` begründet und danach nicht mehr geändert, alle anderen festen Werte unverändert
- [ ] Parameterzahl jeder Architektur ausgegeben (`count_params`) und im Bericht genannt
- [ ] Teil B: Aufgabe und Embedding anhand von Teil A begründet gewählt
- [ ] Teil B: für jedes Modell Durchschnitt und Band über k = 1 bis 9, kleinstes k je Modellpaar im Bericht genannt
- [ ] `rq2_results.csv` erzeugt, Schema eingehalten
- [ ] Jede Abbildung im Bericht lässt sich aus diesem Notebook erzeugen
- [ ] Abgabe: Bericht (PDF), Notebook, CSV, Beitragsnachweis, KI-Hinweis